# 08K: Development foreground patch correspondence

This notebook tests one final 2D improvement to cross-view object identity. It keeps the existing LocateAnything, SAM 2.1, DINOv2-small and P1 pipeline fixed. It does not change the 2,000-scene experiment or use calibration or internal-lock data.

The comparison uses the sealed 100-scene development screen. The current global DINO matching is compared with reciprocal foreground patches, dustbin Sinkhorn patch transport, and an order-free track assembly using the same transport cost.

Notebook file: `08k-development-patch-correspondence.ipynb`.

## Saved-output evidence copy

This notebook records the completed run for repository review. Its code and markdown match the clean runnable notebook. Private scene visuals and identifier-bearing output were removed. Safe aggregate tables and charts were retained.

Run stage: `08K Kaggle Version 3`  
Clean notebook SHA-256: `4f6784d1c07b0a785c3e792d6bd075a202a9f584a5fc95c31a388309abf15b9a`  
Downloaded saved-output SHA-256: `de144b50a36f8d39cfacf811369f419fcdaa7750fed5d1205e53f1cf517153c8`


## Scientific boundary

DINO patch tokens are selected only where the automatic SAM mask covers at least half of a patch. Global and patch distances receive equal weight. Every candidate keeps the existing unmatched cost of 0.35. Sinkhorn uses an epsilon of 0.05, 30 iterations and an explicit dustbin state.

The primary measure is cross-view identity-link F1. Average precision, top-1 accuracy, recall at two, track purity, identity switches, linked coverage, singleton rate, runtime and memory are reported alongside it. The promotion rule rejects gains caused by track fragmentation. There is no arbitrary accuracy gate.

## Runtime and source discovery

In [1]:
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile
import json
import os
import shutil
import stat
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
PACKAGE_ROOT = Path('/tmp/identity-patch-packages')
PREDICTION_OUTPUT = WORKING_ROOT / 'identity-patch-predictions'
EVALUATION_OUTPUT = WORKING_ROOT / 'identity-patch-evaluation'
PYTHON = Path(sys.executable)
EXPECT_RECOVERY = False
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

mount_roots = []
for pattern in ('*', 'datasets/*/*', 'datasets/*/*/versions/*', 'notebooks/*/*', 'notebooks/*/*/versions/*'):
    mount_roots.extend(path.resolve() for path in INPUT_ROOT.glob(pattern) if path.is_dir())
mount_roots = sorted(set(mount_roots))
source_archives = sorted({
    path for mount in mount_roots
    for path in mount.rglob('identity-patch-association-source-*.zip')
    if path.is_file()
})
if len(source_archives) > 1:
    raise ValueError(f'Expected at most one patch-association source archive, found {len(source_archives)}')

if source_archives:
    extraction = WORKING_ROOT / 'identity-patch-source'
    temporary = extraction.with_name(extraction.name + '.incomplete')
    if extraction.exists() or temporary.exists():
        raise FileExistsError('Source extraction destination is not empty')
    temporary.mkdir(parents=True)
    with ZipFile(source_archives[0]) as handle:
        entries = handle.infolist()
        names = [entry.filename for entry in entries]
        if not entries or len(names) != len(set(names)) or handle.testzip() is not None:
            raise ValueError('Source archive is empty, duplicated or corrupt')
        for entry in entries:
            relative = PurePosixPath(entry.filename)
            mode = entry.external_attr >> 16
            if entry.is_dir() or relative.is_absolute() or any(part in ('', '.', '..') for part in relative.parts) or stat.S_ISLNK(mode):
                raise ValueError('Source archive contains an unsafe path')
            destination = temporary.joinpath(*relative.parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with handle.open(entry) as source_stream, destination.open('xb') as target_stream:
                shutil.copyfileobj(source_stream, target_stream, length=1024 * 1024)
    temporary.replace(extraction)
    search_roots = [extraction]
    source_label = source_archives[0].name
else:
    search_roots = mount_roots
    source_label = 'expanded Kaggle dataset'

source_candidates = sorted({
    path.parents[3]
    for root in search_roots
    for path in root.rglob('patch_correspondence.py')
    if (path.parents[3] / 'scripts/run_identity_patch_association.py').is_file()
})
if len(source_candidates) != 1:
    raise ValueError(f'Expected one source root, found {len(source_candidates)}')
SOURCE = source_candidates[0]
sys.path.insert(0, str(SOURCE / 'scripts'))
from kaggle_mounts import find_prediction, find_recovery, materialize_sealed_resource
from scoring_comparison_runtime import run_bounded, tracking_credentials
print(json.dumps({'event': 'source_ready', 'source': source_label}, sort_keys=True))

{"event": "source_ready", "source": "expanded Kaggle dataset"}


## Method references

The reciprocal comparison follows the mutual-nearest-neighbour idea used by Best-Buddies Similarity. The transport comparison combines foreground-weighted semantic correspondence with the dustbin assignment pattern used for partial matching. DINOv2-small remains the frozen visual representation used by the current project.

- Dekel et al., *Best-Buddies Similarity for Robust Template Matching*, CVPR 2015.
- Liu et al., *Semantic Correspondence as an Optimal Transport Problem*, CVPR 2020.
- Sarlin et al., *SuperGlue: Learning Feature Matching with Graph Neural Networks*, CVPR 2020.
- Oquab et al., *DINOv2: Learning Robust Visual Features without Supervision*, 2023.

## Dependency and contract checks

The notebook keeps Kaggle's CUDA-compatible PyTorch build. It installs only pinned add-ons into a run-local folder, compiles the relevant modules and runs the focused unit and workflow-contract tests before loading the model.

In [2]:
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
    '--retries', '1', '--timeout', '60', '--only-binary=:all:',
    '--upgrade', '--target', str(PACKAGE_ROOT),
    '-r', str(SOURCE / 'requirements/identity-patch-association.txt'),
], check=True, timeout=600)
os.environ['PYTHONPATH'] = os.pathsep.join([
    str(SOURCE / 'src'), str(PACKAGE_ROOT), os.environ.get('PYTHONPATH', '')
]).rstrip(os.pathsep)
os.environ['MPLCONFIGDIR'] = str(WORKING_ROOT / 'matplotlib')
sys.path.insert(0, str(PACKAGE_ROOT))
RUNNER = SOURCE / 'scripts/run_identity_patch_association.py'
test_env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
subprocess.run([
    str(PYTHON), '-m', 'pytest', '-p', 'no:cacheprovider', '-q',
    str(SOURCE / 'tests/test_patch_correspondence.py'),
    str(SOURCE / 'tests/test_identity_patch_association_contract.py'),
], check=True, cwd=WORKING_ROOT, env=test_env, timeout=300)
for source_file in (
    SOURCE / 'src/visual_peer_inspector/representation/patch_correspondence.py',
    SOURCE / 'src/visual_peer_inspector/workflows/identity_patch_association_study.py',
    RUNNER,
):
    compile(source_file.read_text(encoding='utf-8'), str(source_file), 'exec')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.0/125.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.8/780.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
ydata-profiling 4.18.4 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator

.............                                                            [100%]
13 passed in 2.05s


## Locked development inputs

In [3]:
MODEL_INPUT = materialize_sealed_resource(
    INPUT_ROOT,
    kind='model-input',
    archive_name='scoring-comparison-model-input.zip',
    archive_sha256='6d6b45aa211ea595524729123870307ee5295ada48e6fd8f1898935f477b6901',
    destination=WORKING_ROOT / 'identity-patch-model-resource',
    expected_lock_sha256='9baa53454d2482caf8240e7a8b591a6e9e9646dd160c5b3abdc8bd17265ebda1',
)
from hashlib import sha256
model_lock_sha256 = sha256((MODEL_INPUT / 'artifact_lock.json').read_bytes()).hexdigest()
SCORING = find_prediction(INPUT_ROOT, expected_input_lock_sha256=model_lock_sha256)
if not (SCORING / 'feature_lock.json').is_file():
    raise FileNotFoundError('Attach the completed 08G output with its feature lock')
run_bounded([
    str(PYTHON), str(RUNNER), 'preflight',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(PREDICTION_OUTPUT),
], WORKING_ROOT / 'identity-patch-preflight.log', idle_seconds=300, total_seconds=900)

{'scene_count': 100, 'scenes': ['[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected identifier removed]', '[protected iden

## Checked recovery plan

Every completed scene is written atomically with a checksum receipt. Recovery is accepted only when the source, inputs, dependencies and protocol match exactly. A fresh run ignores unrelated recovery files mounted by Kaggle.

In [4]:
RECOVERY = find_recovery(INPUT_ROOT) if EXPECT_RECOVERY else None
if EXPECT_RECOVERY and RECOVERY is None:
    raise FileNotFoundError('This continuation requires one verified recovery archive and manifest')
recovery_arguments = [] if RECOVERY is None else [
    '--recovery-bundle', str(RECOVERY[0]),
    '--recovery-manifest', str(RECOVERY[1]),
]
print(json.dumps({
    'event': 'recovery_plan',
    'expected': EXPECT_RECOVERY,
    'attached': RECOVERY is not None,
}, sort_keys=True))

{"attached": false, "event": "recovery_plan", "expected": false}


## Lock label-free identity predictions

One GPU extracts DINOv2-small patch tokens from the eight fixed views. The existing global embeddings and anomaly scores are reused. A 15-minute silent limit stops a stalled process and a four-hour total limit protects GPU time. Completed scene checkpoints remain reusable after an operational interruption.

In [5]:
run_bounded([
    str(PYTHON), str(RUNNER), 'predict',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(PREDICTION_OUTPUT), *recovery_arguments,
], WORKING_ROOT / 'identity-patch-prediction.log', idle_seconds=900, total_seconds=14400)
run_bounded([
    str(PYTHON), str(RUNNER), 'verify',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(PREDICTION_OUTPUT),
], WORKING_ROOT / 'identity-patch-prediction-verification.log', idle_seconds=300, total_seconds=600)
display(json.loads((PREDICTION_OUTPUT / 'run_status.json').read_text()))
display(json.loads((PREDICTION_OUTPUT / 'progress.json').read_text()))

{'event': 'runtime_ready', 'gpu': 'Tesla T4', 'visible_gpus': 2, 'devices_used': 1, 'output_free_bytes': 20940447744}
{"event": "model_load_start", "revision": "ed25f3a31f01632728cabb09d1542f84ab7b0056"}
{'event': 'model_download_start', 'revision': 'ed25f3a31f01632728cabb09d1542f84ab7b0056'}

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 6 files: 100%|██████████| 6/6 [00:06<00:00,  1.13s/it]
{'event': 'model_download_complete', 'snapshot': 'ed25f3a31f01632728cabb09d1542f84ab7b0056'}

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 2493.50it/s]
{'event': 'model_cpu_ready'}
{'event': 'model_cuda_ready', 'allocated_bytes': 88226304}
{"event": "model_ready"}
{"event": "scene_complete", "scene_number": 1, "total": 100}
{"event": "scene_complete", "scene_number": 2, "total": 100}
{"event": "scene_complete", "scene_number": 3, "t

{'scene_count': 100, 'status': 'complete'}

{'complete_scenes': 100, 'expected_scenes': 100}

## Development-only evaluation

The evaluator package is opened only after the prediction lock exists and verifies. It measures identity quality and odd-one-out ranking without changing any prediction.

In [6]:
EVALUATOR = materialize_sealed_resource(
    INPUT_ROOT,
    kind='evaluator-only',
    archive_name='scoring-comparison-evaluator-only.zip',
    archive_sha256='86ac512005af5fdceb64c0fb666998832b8ee08ef96d1fccca23c6e35f420c12',
    destination=WORKING_ROOT / 'identity-patch-evaluator-resource',
    expected_lock_sha256='8012cd20c63b3788f861f2ba991b3181a0c1666f26e176beeee370228c8500c3',
)
run_bounded([
    str(PYTHON), str(RUNNER), 'evaluate',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--predictions', str(PREDICTION_OUTPUT), '--evaluator', str(EVALUATOR),
    '--output', str(EVALUATION_OUTPUT),
], WORKING_ROOT / 'identity-patch-evaluation.log', idle_seconds=900, total_seconds=7200)
run_bounded([
    str(PYTHON), str(RUNNER), 'verify',
    '--model-input', str(MODEL_INPUT), '--scoring', str(SCORING),
    '--output', str(EVALUATION_OUTPUT),
], WORKING_ROOT / 'identity-patch-evaluation-verification.log', idle_seconds=300, total_seconds=600)

{"event": "scene_evaluated", "scene_number": 1, "total": 100}
{"event": "scene_evaluated", "scene_number": 2, "total": 100}
{"event": "scene_evaluated", "scene_number": 3, "total": 100}
{"event": "scene_evaluated", "scene_number": 4, "total": 100}
{"event": "scene_evaluated", "scene_number": 5, "total": 100}
{"event": "scene_evaluated", "scene_number": 6, "total": 100}
{"event": "scene_evaluated", "scene_number": 7, "total": 100}
{"event": "scene_evaluated", "scene_number": 8, "total": 100}
{"event": "scene_evaluated", "scene_number": 9, "total": 100}
{"event": "scene_evaluated", "scene_number": 10, "total": 100}
{"event": "scene_evaluated", "scene_number": 11, "total": 100}
{"event": "scene_evaluated", "scene_number": 12, "total": 100}
{"event": "scene_evaluated", "scene_number": 13, "total": 100}
{"event": "scene_evaluated", "scene_number": 14, "total": 100}
{"event": "scene_evaluated", "scene_number": 15, "total": 100}
{"event": "scene_evaluated", "scene_number": 16, "total": 100}
{

## Results and decision

In [7]:
import pandas as pd
for file_name in (
    'primary_candidate_ranking.csv',
    'patch_association_summary.csv',
    'paired_asset_intervals.csv',
    'identity_coverage_vs_error.csv',
    'patch_fallback_summary.csv',
    'category_summary.csv',
    'family_summary.csv',
):
    print(file_name)
    display(pd.read_csv(EVALUATION_OUTPUT / 'shareable' / file_name))
display(json.loads((EVALUATION_OUTPUT / 'shareable/development_decision.json').read_text()))

primary_candidate_ranking.csv


,configuration_id,family,view_count,method,scenes,asset_groups,ap,top1,recall_at_2,auroc,...,identity_switch_rate,track_completeness,linked_node_fraction,singleton_fraction,mean_track_support,mean_edge_margin,mean_patch_match_fraction,mean_patch_entropy,winner_support,winner_purity
0,current_hungarian_035,global_baseline,3,P1_current,100,27,0.319980,0.23,0.295,0.305462,...,0.903500,0.735278,0.868418,0.231661,2.376201,NaN,NaN,NaN,2.15,0.476667
1,sinkhorn_hungarian_035,foreground_patch_correspondence,3,P1_current,100,27,0.290569,0.20,0.240,0.298190,...,0.913833,0.683889,0.774982,0.363147,2.110190,0.032582,0.890639,0.825871,1.82,0.460000
2,sinkhorn_consensus_035,foreground_patch_correspondence,3,P1_current,100,27,0.290312,0.20,0.260,0.293360,...,0.924167,0.675556,0.816625,0.306274,2.166810,0.032582,0.890639,0.825871,1.88,0.448333
3,reciprocal_hungarian_035,foreground_patch_correspondence,3,P1_current,100,27,0.292294,0.22,0.250,0.302475,...,0.900000,0.627222,0.750045,0.397817,2.034329,0.036735,0.378121,0.000000,1.72,0.480000


patch_association_summary.csv


,configuration_id,family,view_count,method,scenes,asset_groups,ap,top1,recall_at_2,auroc,...,identity_switch_rate,track_completeness,linked_node_fraction,singleton_fraction,mean_track_support,mean_edge_margin,mean_patch_match_fraction,mean_patch_entropy,winner_support,winner_purity
0,current_hungarian_035,global_baseline,2,P1_current,100,27,0.324845,0.24,0.300,0.319875,...,0.790333,0.517083,0.806968,0.257424,1.722576,NaN,NaN,NaN,1.56,0.650000
1,current_hungarian_035,global_baseline,2,P2_consensus,100,27,0.316679,0.22,0.310,0.308625,...,0.790333,0.517083,0.806968,0.257424,1.722576,NaN,NaN,NaN,1.64,0.590000
2,current_hungarian_035,global_baseline,2,forward_02_cov70,100,27,0.335833,0.27,0.325,0.323417,...,0.790333,0.517083,0.806968,0.257424,1.722576,NaN,NaN,NaN,1.59,0.610000
3,current_hungarian_035,global_baseline,3,P1_current,100,27,0.319980,0.23,0.295,0.305462,...,0.903500,0.735278,0.868418,0.231661,2.376201,NaN,NaN,NaN,2.15,0.476667
4,current_hungarian_035,global_baseline,3,P2_consensus,100,27,0.316694,0.21,0.290,0.313333,...,0.903500,0.735278,0.868418,0.231661,2.376201,NaN,NaN,NaN,2.15,0.455000
5,current_hungarian_035,global_baseline,3,forward_02_cov70,100,27,0.339835,0.24,0.355,0.335573,...,0.903500,0.735278,0.868418,0.231661,2.376201,NaN,NaN,NaN,2.10,0.426667
6,current_hungarian_035,global_baseline,5,P1_current,100,27,0.275236,0.19,0.295,0.301190,...,0.951667,0.664667,0.930616,0.179067,3.721607,NaN,NaN,NaN,2.88,0.301833
7,current_hungarian_035,global_baseline,5,P2_consensus,100,27,0.254242,0.15,0.270,0.274824,...,0.951667,0.664667,0.930616,0.179067,3.721607,NaN,NaN,NaN,2.94,0.299833
8,current_hungarian_035,global_baseline,5,forward_02_cov70,100,27,0.265845,0.17,0.280,0.299342,...,0.951667,0.664667,0.930616,0.179067,3.721607,NaN,NaN,NaN,3.10,0.313333
9,reciprocal_hungarian_035,foreground_patch_correspondence,2,P1_current,100,27,0.344512,0.26,0.320,0.348333,...,0.801667,0.511667,0.692000,0.393400,1.586600,0.029845,0.350988,0.000000,1.42,0.655000


paired_asset_intervals.csv


...

category_summary.csv


,configuration_id,family,view_count,method,category,ap,top1,identity_link_f1
0,current_hungarian_035,global_baseline,2,P1_current,apple,0.000000,0.000000,0.000000
1,current_hungarian_035,global_baseline,2,P1_current,bottle,0.145833,0.000000,0.083333
2,current_hungarian_035,global_baseline,2,P1_current,bowl,0.500000,0.333333,0.111111
3,current_hungarian_035,global_baseline,2,P1_current,bread,0.098214,0.000000,0.083333
4,current_hungarian_035,global_baseline,2,P1_current,bunny,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
967,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,monkey,0.055556,0.000000,0.139040
968,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,mouse,0.122222,0.000000,0.120430
969,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,orange,0.277778,0.333333,0.152836
970,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,pig,0.500000,0.333333,0.193791


family_summary.csv


,configuration_id,family,view_count,method,families,ap,top1,identity_link_f1
0,current_hungarian_035,global_baseline,2,P1_current,bump,0.287879,0.181818,0.088889
1,current_hungarian_035,global_baseline,2,P1_current,crack,0.362745,0.294118,0.075630
2,current_hungarian_035,global_baseline,2,P1_current,crack+disc,0.208333,0.250000,0.125000
3,current_hungarian_035,global_baseline,2,P1_current,crack+frac,0.195982,0.125000,0.125992
4,current_hungarian_035,global_baseline,2,P1_current,defr,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
391,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,frac,0.370175,0.210526,0.192626
392,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,miss,0.462500,0.375000,0.329436
393,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,miss+tran,0.300000,0.500000,0.244152
394,sinkhorn_hungarian_035,foreground_patch_correspondence,5,forward_02_cov70,rota,0.000000,0.000000,0.118026


{'baseline': 'current_hungarian_035',
 'prediction_producer': 'dinov2-small-foreground-patch-correspondence',
 'primary_route': 'automatic masks, predicted identity, three views',
 'primary_scorer': 'P1_current',
 'promotion': 'No candidate passed the joint identity, ranking and anti-fragmentation rule.',
 'protected_data_used': False,
 'review_candidate': 'sinkhorn_hungarian_035',
 'scientific_result': True,
 'selected_configuration': 'current_hungarian_035',
 'split_changed': False,
 'status': 'retain_current'}

## Figures and private examples

Summary figures compare identity quality, odd-one-out accuracy, and the wrong-link rate as lower-confidence links are removed. The private gallery shows fixed improvements, regressions and representative cases. Red marks the selected identity, green marks the reference unusual object, and blue marks other automatic candidates.

In [8]:
from IPython.display import Image as NotebookImage, display
for figure in sorted((EVALUATION_OUTPUT / 'shareable/figures').glob('*.png')):
    print(figure.name)
    display(NotebookImage(filename=str(figure)))
gallery = json.loads((EVALUATION_OUTPUT / 'private/gallery_manifest.json').read_text())
for item in gallery['examples']:
    print(item['file'], item['selection_reason'])
    display(NotebookImage(filename=str(EVALUATION_OUTPUT / 'private/figures' / item['file'])))

identity_coverage_vs_error.png


<IPython.core.display.Image object>

identity_vs_top1.png


<IPython.core.display.Image object>

primary_top1_comparison.png


...

<IPython.core.display.Image object>

identity_example_11.png fixed_hash_sample


<IPython.core.display.Image object>

identity_example_12.png fixed_hash_sample


<IPython.core.display.Image object>

## Post-compute ClearML upload

ClearML runs only after the scientific outputs are sealed. It receives aggregate tables, decisions and summary figures. Private images, labels, per-scene records, model inputs and credentials are excluded. Tracking failure cannot trigger another GPU run.

In [9]:
if tracking_credentials():
    try:
        subprocess.run([
            str(PYTHON), '-m', 'pip', 'install', '--retries', '1', '--timeout', '30',
            '--only-binary=:all:', '--upgrade', '--target', str(PACKAGE_ROOT),
            '-r', str(SOURCE / 'requirements/identity-patch-association-tracking.txt'),
        ], check=True, timeout=180)
        subprocess.run([
            str(PYTHON), str(SOURCE / 'scripts/log_identity_patch_association.py'),
            '--evidence', str(EVALUATION_OUTPUT),
        ], check=True, timeout=300)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        print(f'ClearML upload remains retryable: {type(error).__name__}')
else:
    print('ClearML secrets are unavailable. Scientific outputs are complete and unchanged.')

ClearML secrets are unavailable. Scientific outputs are complete and unchanged.


## Download package

In [10]:
RESULT_ZIP = WORKING_ROOT / 'identity-patch-study-results.zip'
with ZipFile(RESULT_ZIP, 'w', compression=ZIP_DEFLATED, compresslevel=6) as handle:
    for root in (PREDICTION_OUTPUT, EVALUATION_OUTPUT):
        for path in sorted(root.rglob('*')):
            if path.is_file():
                handle.write(path, Path(root.name) / path.relative_to(root))
    for path in sorted(WORKING_ROOT.glob('identity-patch-*.log')):
        handle.write(path, Path('logs') / path.name)
with ZipFile(RESULT_ZIP) as handle:
    if handle.testzip() is not None:
        raise ValueError('The result archive failed its integrity check')
print(json.dumps({
    'result_zip': str(RESULT_ZIP),
    'size_bytes': RESULT_ZIP.stat().st_size,
    'prediction_output': str(PREDICTION_OUTPUT),
    'evaluation_output': str(EVALUATION_OUTPUT),
    'decision': json.loads((EVALUATION_OUTPUT / 'shareable/development_decision.json').read_text()),
}, indent=2, sort_keys=True))

{
  "decision": {
    "baseline": "current_hungarian_035",
    "prediction_producer": "dinov2-small-foreground-patch-correspondence",
    "primary_route": "automatic masks, predicted identity, three views",
    "primary_scorer": "P1_current",
    "promotion": "No candidate passed the joint identity, ranking and anti-fragmentation rule.",
    "protected_data_used": false,
    "review_candidate": "sinkhorn_hungarian_035",
    "scientific_result": true,
    "selected_configuration": "current_hungarian_035",
    "split_changed": false,
    "status": "retain_current"
  },
  "evaluation_output": "/kaggle/working/identity-patch-evaluation",
  "prediction_output": "/kaggle/working/identity-patch-predictions",
  "result_zip": "/kaggle/working/identity-patch-study-results.zip",
  "size_bytes": 4532295
}


## Interpretation rule

Promote a candidate only if the paired asset-group interval improves identity-link F1 and average precision, top-1 and recall at two do not regress, and linked coverage and singleton rate remain acceptable. Otherwise retain the current Hungarian method and use the confidence evidence to identify cases that need optional review.